# 16. Lecke - Skálázható ügynökök telepítése a Microsoft Foundry-val

Ebben a jegyzetfüzetben egy **élesítésre kész ügyféltámogató ügynököt** építesz a fiktív **Contoso** vállalat számára. A korábbi leckékkel ellentétben itt nem az ügynök érvelési ciklusa a lényeg — hanem minden, ami *körülötte* van, ami biztonságossá teszi az ügynököt a méretezett működéshez:

1. **Eszköz hívás** — rendelés lekérdezés és jegy létrehozás.
2. **RAG** — szabályzati válaszok egy tudásbázisból.
3. **Memória** — az ügyfél emlékezése a beszélgetések során.
4. **Modell irányítás** — egyszerű kérések egy kis modellhez, bonyolultak egy nagy modellhez.
5. **Válasz gyorsítótárazás** — ismételt kérdések kiszolgálása modellhívás nélkül.
6. **Humán jóváhagyás** — küszöbérték feletti visszatérítések megállítása aláírásra.
7. **Értékelő kapu** — egy offline teszthalmaz, amely blokkolja a rossz kiadást.
8. **Megfigyelhetőség** — OpenTelemetry követés minden kérés körül.

Minden szakasz önálló és futtatható. Olvasd el minden sorát — az élesítési elemek szándékosan kicsik.


## Beállítás

A jegyzetfüzet futtatása előtt győződjön meg arról, hogy rendelkezik:

1. **Egy Microsoft Foundry projekttel**, amelyben üzembe helyezett egy chat modellt (pl. `gpt-4.1-mini`).
2. **Be van jelentkezve az Azure CLI-vel** — futtassa a `az login` parancsot a termináljában.
3. **Beállította a szükséges környezeti változókat:**
   - `AZURE_AI_PROJECT_ENDPOINT` — a Microsoft Foundry projekt végpontja.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — az üzembe helyezett modell neve.

A RAG szakasz az **Azure AI Search** szolgáltatást használja, ha az `AZURE_SEARCH_SERVICE_ENDPOINT` és az `AZURE_SEARCH_API_KEY` be van állítva, különben memóriabeli keresést alkalmaz, így a jegyzetfüzet kereső erőforrás nélkül is fut.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import re
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME in your .env file."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential(),
)
print("Foundry client ready.")

## 1. Eszközök

A gyártási eszközök valódi munkát végeznek valós rendszereken. Itt egy egyszerű Python függvényekkel szimulálunk egy rendelési adatbázist és egy jegykezelő rendszert. Az `@tool` dekorátor teszi elérhetővé őket az ügynök számára.

Vegyük észre, hogy az `issue_refund` az `approval_mode="always_require"` beállítást használja bizonyos küszöbérték feletti visszatérítésekhez — ez a később alkalmazott, emberi beavatkozást igénylő primitív funkció.


In [ ]:
# Simulated backend systems (in production these are API calls behind scoped identities).
ORDERS = {
    "A1001": {"status": "shipped", "total": 42.00, "eta": "2 days"},
    "A1002": {"status": "processing", "total": 128.50, "eta": "5 days"},
    "A1003": {"status": "delivered", "total": 19.99, "eta": "delivered"},
}
TICKETS: list[dict] = []
REFUND_APPROVAL_THRESHOLD = 50.0


@tool(approval_mode="never_require")
def get_order_status(order_id: Annotated[str, "The customer's order ID, e.g. A1001"]) -> str:
    """Look up the status of a customer order."""
    order = ORDERS.get(order_id.upper())
    if not order:
        return f"No order found with ID {order_id}."
    return (
        f"Order {order_id.upper()}: status={order['status']}, "
        f"total=${order['total']:.2f}, eta={order['eta']}."
    )


@tool(approval_mode="never_require")
def open_ticket(
    subject: Annotated[str, "Short subject line for the support ticket"],
    details: Annotated[str, "Full description of the customer's issue"],
) -> str:
    """Open a support ticket for issues that need human follow-up."""
    ticket_id = f"T{1000 + len(TICKETS) + 1}"
    TICKETS.append({"id": ticket_id, "subject": subject, "details": details})
    return f"Ticket {ticket_id} opened: {subject}"


def refund_needs_approval(amount: float) -> bool:
    """Refunds above the threshold require a human approver."""
    return amount > REFUND_APPROVAL_THRESHOLD


@tool(approval_mode="always_require")
def issue_refund(
    order_id: Annotated[str, "The order to refund"],
    amount: Annotated[float, "Refund amount in USD"],
) -> str:
    """Issue a refund. Execution pauses for human approval before it runs."""
    return f"Refund of ${amount:.2f} issued for order {order_id.upper()}."


print("Tools defined.")

## 2. RAG — Szabályzat Tudásbázis

A szabályzati kérdésekre („mi a visszaküldési időszabályod?”) hatósági forrásból kell válaszolni, nem a modell memóriájából. Egy kis tudásbázist csomagolunk keresőeszközként.

Üzemeltetésben ez az **Azure AI Search**; itt egy memóriabeli kulcsszavas keresést adunk meg, hogy a jegyzetfüzet bárhol fusson, és automatikusan átvált az Azure AI Search-re, ha a környezeti változók jelen vannak.


In [ ]:
KNOWLEDGE_BASE = {
    "returns": "Contoso accepts returns within 30 days of delivery for a full refund. Items must be unused and in original packaging.",
    "shipping": "Standard shipping takes 3-5 business days. Express shipping (1-2 days) is available at checkout for an extra fee.",
    "warranty": "All Contoso electronics carry a 12-month limited warranty covering manufacturing defects.",
    "refund_policy": "Refunds are processed to the original payment method within 5 business days of approval. Refunds over $50 require a supervisor's approval.",
}


def _in_memory_search(query: str) -> str:
    q = query.lower()
    hits = [text for key, text in KNOWLEDGE_BASE.items() if key.replace("_", " ") in q or key in q]
    if not hits:
        # crude keyword fallback so the tool still returns something useful
        hits = [text for text in KNOWLEDGE_BASE.values() if any(w in text.lower() for w in q.split())]
    return "\n".join(hits) if hits else "No matching policy found."


def _azure_search(query: str) -> str:
    from azure.core.credentials import AzureKeyCredential
    from azure.search.documents import SearchClient

    client = SearchClient(
        endpoint=os.environ["AZURE_SEARCH_SERVICE_ENDPOINT"],
        index_name=os.getenv("AZURE_SEARCH_INDEX_NAME", "contoso-policies"),
        credential=AzureKeyCredential(os.environ["AZURE_SEARCH_API_KEY"]),
    )
    results = client.search(search_text=query, top=3)
    return "\n".join(r.get("content", "") for r in results) or "No matching policy found."


USE_AZURE_SEARCH = bool(os.getenv("AZURE_SEARCH_SERVICE_ENDPOINT") and os.getenv("AZURE_SEARCH_API_KEY"))


@tool(approval_mode="never_require")
def search_policies(query: Annotated[str, "The policy question to look up"]) -> str:
    """Search Contoso support policies to answer customer questions."""
    if USE_AZURE_SEARCH:
        return _azure_search(query)
    return _in_memory_search(query)


print(f"RAG ready. Using {'Azure AI Search' if USE_AZURE_SEARCH else 'in-memory search'}.")

## 3. Memória

Egy támogatási ügynök, aki elfelejti, kivel beszél, rossz támogatási ügynök. Minden ügyfélhez tartunk egy apró profiltárolót, és egy rövid összefoglalót injektálunk az ügynök utasításaiba. Éles környezetben ez egy memóriaszolgáltatás (lásd 13. lecke); itt egy szótár teszi a mintát láthatóvá.


In [ ]:
CUSTOMER_MEMORY: dict[str, dict] = {
    "cust-42": {"name": "Dana", "tier": "enterprise", "recent_order": "A1002"},
    "cust-99": {"name": "Sam", "tier": "standard", "recent_order": "A1003"},
}


def memory_context(customer_id: str) -> str:
    profile = CUSTOMER_MEMORY.get(customer_id)
    if not profile:
        return "This is a new customer with no history."
    return (
        f"Customer {profile['name']} ({profile['tier']} tier). "
        f"Most recent order: {profile['recent_order']}."
    )


print(memory_context("cust-42"))

## 4. és 5. Modell irányítás és válasz gyorsítótárazása

Két költséghatékony beavatkozás egyetlen kéréskezelőbe kötve:

- **Irányítás**: egy olcsó heurisztikus osztályozó dönt arról, hogy egy kérésnek a kicsi vagy a nagy modellre van-e szüksége.
- **Gyorsítótárazás**: normált ismétlődő kérdéseket gyorsítótárból szolgálunk ki, modell hívás nélkül.

Az osztályozó itt szándékosan egyszerű. Üzembe helyezéskor érdemes forgalom alapján validálni, és helyettesíteni a Foundry Modell Irányítójával.


In [ ]:
SMALL_MODEL = os.getenv("AZURE_AI_SMALL_MODEL", model)   # e.g. gpt-4.1-mini
LARGE_MODEL = os.getenv("AZURE_AI_LARGE_MODEL", model)   # e.g. gpt-4.1

response_cache: dict[str, str] = {}
route_counters = {"small": 0, "large": 0, "cache": 0}


def normalize(query: str) -> str:
    return re.sub(r"\s+", " ", query.lower().strip())


COMPLEX_SIGNALS = ("refund", "cancel", "complaint", "escalate", "broken", "wrong", "why")


def is_simple(query: str) -> bool:
    """Route complex or high-stakes requests to the large model; everything else to the small one."""
    q = query.lower()
    if any(signal in q for signal in COMPLEX_SIGNALS):
        return False
    return len(q.split()) <= 20


def choose_model(query: str) -> str:
    return SMALL_MODEL if is_simple(query) else LARGE_MODEL


print(f"Small model: {SMALL_MODEL} | Large model: {LARGE_MODEL}")

## 6 & 8. Az ügynök, emberi jóváhagyás és megfigyelhetőség

Most az ügynököt a fenti eszközökből állítjuk össze, és minden kérést egy OpenTelemetry span-be csomagolunk. A `handle_support_request` függvény a termelési kérések kezelője: cache → útvonal → trace → futtatás → cache.

Az emberi jóváhagyást a keretrendszer kezeli: mivel az `issue_refund` `approval_mode="always_require"` értékű, a futtatás szünetel, és megjeleníti a jóváhagyási kérelmet, amelyet kifejezetten megoldunk.


In [ ]:
# Tracing: use the Agent Framework tracer if available, else a no-op so the notebook runs anywhere.
try:
    from agent_framework.observability import get_tracer
    tracer = get_tracer()
except Exception:  # observability extras not installed
    from contextlib import contextmanager

    class _NoopSpan:
        def set_attribute(self, *_args, **_kwargs):
            pass

    class _NoopTracer:
        @contextmanager
        def start_as_current_span(self, _name):
            yield _NoopSpan()

    tracer = _NoopTracer()


SUPPORT_INSTRUCTIONS = (
    "You are Contoso's customer support agent. Be concise, friendly, and accurate. "
    "Use search_policies for policy questions, get_order_status for orders, "
    "open_ticket when a human needs to follow up, and issue_refund for refunds. "
    "Never invent policy details."
)

support_agent = provider.as_agent(
    name="ContosoSupportAgent",
    instructions=SUPPORT_INSTRUCTIONS,
    tools=[get_order_status, open_ticket, search_policies, issue_refund],
)


async def handle_support_request(query: str, customer_id: str) -> str:
    # 1. Serve from cache when we can.
    key = normalize(query)
    if key in response_cache:
        route_counters["cache"] += 1
        return response_cache[key]

    # 2. Route by complexity to control cost.
    chosen_model = choose_model(query)
    route_counters["small" if chosen_model == SMALL_MODEL else "large"] += 1

    # 3. Add per-customer memory to the prompt.
    context = memory_context(customer_id)
    prompt = f"[Customer context: {context}]\n\n{query}"

    # 4. Run inside a trace span for observability.
    with tracer.start_as_current_span("support_request") as span:
        span.set_attribute("customer.id", customer_id)
        span.set_attribute("routed.model", chosen_model)
        response = await support_agent.run(prompt, model=chosen_model)

    text = response.text
    response_cache[key] = text
    return text


print("Support agent assembled.")

In [ ]:
# Try a few requests. The first is simple (small model), the second is a refund (large model + approval path).
print(await handle_support_request("What is your return window?", "cust-99"))
print("---")
print(await handle_support_request("Where is my order A1002?", "cust-42"))
print("---")
# Repeat the first question -> served from cache.
print(await handle_support_request("What is your return window?", "cust-99"))
print("---")
print("Routing counters:", route_counters)

## 7. Értékelési kapu

Ez a kiadási kapu a leckéből: egy offline tesztkészlet pontozza az ügynököt, és a telepítés csak akkor folytatódik, ha a teljesítési arány átlépi a küszöböt. Itt az értékelő egy egyszerű kulcsszó-átfedés ellenőrzés, hogy a jegyzet önálló legyen; éles környezetben LLM-bíró vagy egy keretrendszer értékelőt használnál (lásd a 10. leckét).


In [ ]:
TEST_CASES = [
    {"input": "How long do I have to return an item?", "expected": ["30 days", "refund"]},
    {"input": "How fast is standard shipping?", "expected": ["3-5", "business days"]},
    {"input": "What is the status of order A1001?", "expected": ["shipped", "A1001"]},
    {"input": "Do your electronics have a warranty?", "expected": ["12-month", "warranty"]},
]


def score_response(actual: str, expected_keywords: list[str]) -> float:
    actual_l = actual.lower()
    hits = sum(1 for kw in expected_keywords if kw.lower() in actual_l)
    return hits / len(expected_keywords)


async def evaluation_gate(test_cases: list[dict], threshold: float = 0.8) -> bool:
    passed = 0
    for case in test_cases:
        result = await support_agent.run(case["input"])
        s = score_response(result.text, case["expected"])
        status = "PASS" if s >= 0.5 else "FAIL"
        print(f"[{status}] {case['input']}  (score={s:.0%})")
        if s >= 0.5:
            passed += 1
    pass_rate = passed / len(test_cases)
    print(f"\nEvaluation pass rate: {pass_rate:.0%} (gate: {threshold:.0%})")
    return pass_rate >= threshold


gate_passed = await evaluation_gate(TEST_CASES, threshold=0.8)
print("\nDeploy allowed:" , gate_passed)

## Összerakva: Egy Szimulált Kiadás

Az alábbi sejt mutatja az egész ciklust, amit a lecke ismertet: futtassa le az értékelési kaput, és csak akkor „telepítsen”, ha az átmegy. Ez az a minta, amit CI-ben futtatna, mielőtt egy ügynökverziót előléptetne a Foundry Agent Szolgáltatáshoz.


In [ ]:
async def release(test_cases: list[dict]) -> None:
    print("Running pre-deployment evaluation gate...\n")
    if await evaluation_gate(test_cases, threshold=0.8):
        print("\n✅ Gate passed — promoting agent version to the Foundry Agent Service.")
    else:
        print("\n❌ Gate failed — release blocked. Fix the agent and re-run.")


await release(TEST_CASES)

## Összefoglaló

Összeállítottál egy éles használatra alkalmas ügyféltámogató ügynököt, amely minden működési szempontot figyelembe vesz:

- **Eszközök, RAG és memória** adják meg az ügynök képességeit és kontextusát.
- **Modell irányítás és gyorsítótárazás** tartják kordában a késleltetést és a költségeket.
- **Emberi jóváhagyás** védi a nagy kockázatú műveleteket, mint például a nagy összegű visszatérítéseket.
- **Az értékelő kapu** megakadályozza a hibás kiadásokat, mielőtt azok megjelennek.
- **Nyomkövetés** megfigyelhetővé tesz minden kérést.

### Kihívás

Bővítsd ezt az ügynököt:

1. **Több modell támogatása** — adj hozzá egy harmadik, "érvelési" réteget, és irányítsd hozzá a felterjesztéseket/panaszokat.
2. **Értékelő kapuk hozzáadása** — bővítsd a `TEST_CASES`-t visszatérítés jóváhagyási forgatókönyvekkel, és ellenőrizd, hogy a kapu kiszűri-e a regressziókat.
3. **Költségtudatos irányítás hozzáadása** — kövesd nyomon a becsült költséget kérésenként (kicsi vs nagy vs gyorsítótár), és készíts költségjelentést kevert lekérdezések egy adagja után.

A következő leckében az ellenkező utat járjuk be, és az ügynököt teljes egészében a saját gépeden futtatod Microsoft Foundry Local és Qwen segítségével.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Jogi nyilatkozat**:
Ez a dokumentum az AI fordítási szolgáltatás, a [Co-op Translator](https://github.com/Azure/co-op-translator) segítségével készült. Bár az pontosságra törekszünk, kérjük, vegye figyelembe, hogy az automatikus fordítások hibákat vagy pontatlanságokat tartalmazhatnak. Az eredeti dokumentum az anyanyelvén tekintendő hiteles forrásnak. Fontos információk esetén professzionális emberi fordítást javasolunk. Nem vállalunk felelősséget semmilyen félreértésért vagy téves értelmezésért, amely ebből a fordításból ered.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
